# MergeME: Model Merging Techniques for Homogeneous and Heterogeneous MoEs

This Jupyter Notebook implements the key methodologies described in the paper "MergeME: Model Merging Techniques for Homogeneous and Heterogeneous MoEs" (arXiv:2502.00997v3). The paper introduces novel techniques for merging specialized Large Language Models (LLMs) into a unified Mixture-of-Experts (MoE) model, addressing challenges like parameter interference and heterogeneous architectures.

This notebook will demonstrate:
1.  **Homogeneous Model Merging:** Using advanced merging methods like Dare and Ties to mitigate parameter interference in non-FFN layers.
2.  **Perplexity-Based Routing:** Implementing a routing heuristic for MoE without extensive fine-tuning.
3.  **Heterogeneous Model Merging:** A conceptual approach for merging models with different architectures using projector layers.

**Note:** This notebook provides illustrative code based on the paper's descriptions. Actual execution requires a suitable environment with `torch`, `transformers`, `mergekit`, and access to specific Hugging Face models. Due to environment limitations, model loading, actual merging, and evaluation steps are represented conceptually.

In [ ]:
# 1. Setup and Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from mergekit.config import MergeConfiguration, Model
from mergekit.merge import merge_models
import numpy as np
import math

# --- Configuration --- 
# Define placeholder model names for demonstration. 
# In a real scenario, these would be actual Hugging Face model paths.
BASE_MODEL_NAME = "meta-llama/Llama-2-7b-hf"
MATH_EXPERT_NAME = "deepseek-ai/deepseek-math-7b-base"
CODE_EXPERT_NAME = "codellama/CodeLlama-7b-hf"
KNOWLEDGE_EXPERT_NAME = "google/gemma-2b"
MATH_TINYLAMA_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MATH_OLMO_NAME = "allenai/OLMo-1B"

# Parameters from the paper (Appendix A)
LAMBDA_SCALING = 1/3
RETAIN_RATIO_P = 0.8 # 80%
TOP_K_ROUTING = 2 # Top-K experts for MoE routing

print("Setup complete. Placeholder model names defined.")

# --- Hugging Face Login (Optional) ---
# If you plan to pull private models or push merged models to Hugging Face Hub,
# you will need to log in.
# You can generate a token from your Hugging Face settings page (https://huggingface.co/settings/tokens).
# Then, run the following command in your terminal or uncomment and run the code below:
# !huggingface-cli login
# Or programmatically:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE") # Replace with your actual token or use environment variable

print("\nHugging Face login instructions added. Proceed to next sections for model merging.")

: 

## 2. Homogeneous Model Merging (Dare / Ties)

The paper proposes replacing simple unweighted averaging with more advanced merging methods like Dare and Ties to mitigate parameter interference in homogeneous MoE merging. These methods operate on 'task vectors' (difference between expert and base model parameters) by trimming and re-scaling them.

Here, we'll define a `mergekit` configuration that conceptually applies these methods. `mergekit` typically uses a YAML configuration, but we'll represent it programmatically for demonstration.

In [ ]:
# Conceptual function to create a mergekit configuration for homogeneous merging
def create_homogeneous_merge_config(merge_method: str, base_model: str, experts: list):
    """
    Creates a conceptual mergekit configuration for homogeneous model merging.
    
    Args:
        merge_method (str): 'dare_ties' or 'ties_merging' (as per mergekit's capabilities).
        base_model (str): Name of the base model.
        experts (list): List of expert model names.
    
    Returns:
        dict: A dictionary representing the mergekit configuration.
    """
    models_config = []
    # Base model
    models_config.append({"model": base_model, "parameters": {"density": 1.0, "weight": 1.0}})
    
    # Experts
    for expert in experts:
        models_config.append({"model": expert, "parameters": {"density": RETAIN_RATIO_P, "weight": LAMBDA_SCALING}})
        
    # The mergekit library handles the specific logic for 'dare_ties' or 'ties_merging'
    # based on the specified merge method in the config.
    # For non-FFN layers, the paper suggests applying these methods.
    # For FFN layers, they are kept separate in an MoE setup.
    
    # This is a simplified representation. A real mergekit config would be more detailed.
    config = {
        "models": models_config,
        "merge_method": merge_method, # e.g., "dare_ties" or "ties_merging"
        "base_model": base_model,
        "parameters": {
            "int8_tensors": True, # Example parameter
            "tokenizer_source": base_model
        },
        "slices": [
            # Example slice for non-FFN layers (conceptual)
            {"layers": "all", "merge_method": merge_method, "parameters": {"density": RETAIN_RATIO_P, "weight": LAMBDA_SCALING}}
            # FFN layers would be handled separately for MoE, not merged in this way
        ]
    }
    return config

# Example usage for Dare merging
homogeneous_experts = [MATH_EXPERT_NAME, CODE_EXPERT_NAME, KNOWLEDGE_EXPERT_NAME]
dare_config = create_homogeneous_merge_config("dare_ties", BASE_MODEL_NAME, homogeneous_experts)
print("\nDare Merging Configuration (Conceptual):")
print(dare_config)

# Example usage for Ties merging
ties_config = create_homogeneous_merge_config("ties_merging", BASE_MODEL_NAME, homogeneous_experts)
print("\nTies Merging Configuration (Conceptual):")
print(ties_config)

# --- Actual Merging (Conceptual) ---
# In a real scenario, you would load models and perform the merge.
# This part is commented out as it requires actual model files and significant resources.
# try:
#     # Load tokenizer and base model
#     tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
#     base_model_loaded = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.bfloat16)
#     
#     # Prepare models for mergekit (this is a simplified representation)
#     models_to_merge = [
#         Model(path=BASE_MODEL_NAME, parameters={"density": 1.0, "weight": 1.0}),
#         Model(path=MATH_EXPERT_NAME, parameters={"density": RETAIN_RATIO_P, "weight": LAMBDA_SCALING}),
#         Model(path=CODE_EXPERT_NAME, parameters={"density": RETAIN_RATIO_P, "weight": LAMBDA_SCALING}),
#         Model(path=KNOWLEDGE_EXPERT_NAME, parameters={"density": RETAIN_RATIO_P, "weight": LAMBDA_SCALING}),
#     ]
#     
#     # Create a MergeConfiguration object
#     merge_config_obj = MergeConfiguration(
#         models=models_to_merge,
#         merge_method="dare_ties", # or "ties_merging"
#         base_model=BASE_MODEL_NAME,
#         parameters={"int8_tensors": True, "tokenizer_source": BASE_MODEL_NAME},
#         # slices would be defined here for specific layer merging strategies
#     )
#     
#     # Perform the merge
#     # merged_model = merge_models(merge_config_obj, tokenizer=tokenizer)
#     # print("Homogeneous model merging (Dare/Ties) conceptual step complete.")
# except Exception as e:
#     print(f"Error during conceptual model loading/merging: {e}")
#     print("Please ensure `torch`, `transformers`, and `mergekit` are installed and models are accessible.")


## 3. Perplexity-Based Routing Heuristics

The paper proposes a perplexity (PPL) based routing heuristic to reduce reliance on MoE fine-tuning. This method selects experts with the lowest PPL values for a given input sequence, indicating higher confidence.

We'll define a function to calculate PPL and then use it for conceptual routing.

In [ ]:
def calculate_perplexity(model, tokenizer, text: str):
    """
    Calculates the perplexity of a given text using the provided model and tokenizer.
    This is a conceptual implementation. Actual PPL calculation can be resource-intensive.
    """
    if model is None or tokenizer is None:
        # Return a dummy PPL for conceptual demonstration if models are not loaded
        # In a real scenario, this would involve actual model inference.
        return np.random.uniform(50, 200) # Simulate varying perplexity

    try:
        encodings = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        input_ids = encodings.input_ids
        target_ids = input_ids.clone()
        target_ids[:, :-1] = -100 # Mask out the last token for PPL calculation

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss
        
        ppl = torch.exp(neg_log_likelihood).item()
        return ppl
    except Exception as e:
        print(f"Error calculating PPL: {e}. Returning dummy value.")
        return np.random.uniform(50, 200)

def perplexity_routing(input_sequence: str, expert_models: dict, tokenizer):
    """
    Implements perplexity-based routing heuristic.
    Selects top-K experts based on lowest perplexity.
    """
    ppl_scores = {}
    for name, model in expert_models.items():
        ppl = calculate_perplexity(model, tokenizer, input_sequence)
        ppl_scores[name] = ppl
    
    print(f"Perplexity scores for input '{input_sequence[:50]}...': {ppl_scores}")
    
    # Sort experts by perplexity (lower is better)
    sorted_experts = sorted(ppl_scores.items(), key=lambda item: item[1])
    
    # Select top-K experts
    top_k_experts = sorted_experts[:TOP_K_ROUTING]
    
    # Calculate weights using SoftMax on reciprocal of PPL
    reciprocal_ppls = [1 / score for _, score in top_k_experts]
    softmax_weights = torch.softmax(torch.tensor(reciprocal_ppls), dim=0).tolist()
    
    routing_decision = {
        "selected_experts": [{'name': name, 'weight': weight} for (name, _), weight in zip(top_k_experts, softmax_weights)],
        "all_ppl_scores": ppl_scores
    }
    return routing_decision

# --- Conceptual Usage ---
# In a real scenario, you would load actual models here.
# For this demonstration, we'll use None for models and a dummy tokenizer.
dummy_tokenizer = AutoTokenizer.from_pretrained("gpt2") # Using a common tokenizer for demonstration
dummy_expert_models = {
    "Base": None, # Placeholder for loaded base model
    "Math Expert": None, # Placeholder for loaded math expert
    "Code Expert": None, # Placeholder for loaded code expert
    "Knowledge Expert": None # Placeholder for loaded knowledge expert
}

math_query = "What is the integral of x^2 from 0 to 1?"
code_query = "Write a Python function to reverse a string."
general_query = "Who was Marie Curie?"

print("\nPerplexity-Based Routing for Math Query:")
math_routing = perplexity_routing(math_query, dummy_expert_models, dummy_tokenizer)
print(math_routing)

print("\nPerplexity-Based Routing for Code Query:")
code_routing = perplexity_routing(code_query, dummy_expert_models, dummy_tokenizer)
print(code_routing)

print("\nPerplexity-Based Routing for General Query:")
general_routing = perplexity_routing(general_query, dummy_expert_models, dummy_tokenizer)
print(general_routing)


## 4. Heterogeneous Model Merging

The paper introduces a novel framework for merging experts with different architectures into a unified MoE. This involves shared embedding/head layers, projector layers (Proj-in, Proj-out) to unify dimensions, and sequence-level routing.

This section will outline the conceptual steps and components required for such a merge, as `mergekit` primarily focuses on homogeneous merging or specific heterogeneous cases. A full implementation would require custom `torch` modules.

In [ ]:
# Conceptual representation of heterogeneous merging components
class HeterogeneousMoE(torch.nn.Module):
    def __init__(self, base_model_name, expert_names: list, max_hidden_dim: int = 2048):
        super().__init__()
        self.max_hidden_dim = max_hidden_dim
        self.expert_names = expert_names
        
        # 1. Shared Embedding Layer (conceptual)
        # In a real scenario, this would be initialized from averaged embeddings
        # of all experts, potentially with padding for smaller dimensions.
        # For demonstration, we'll use a dummy embedding layer.
        vocab_size = 50257 # Example vocab size (e.g., for gpt2 tokenizer)
        self.shared_embedding = torch.nn.Embedding(vocab_size, max_hidden_dim)
        
        # 2. Expert Decoders and Projectors
        self.experts = torch.nn.ModuleDict()
        self.proj_ins = torch.nn.ModuleDict()
        self.proj_outs = torch.nn.ModuleDict()
        
        # Simulate different expert architectures and dimensions
        # In a real scenario, you'd load actual models and extract their dimensions.
        expert_dims = {
            BASE_MODEL_NAME: 2048,
            CODE_EXPERT_NAME: 2048,
            KNOWLEDGE_EXPERT_NAME: 2048,
            MATH_TINYLAMA_NAME: 2048, # TinyLlama-1.1B has 2048 hidden dim
            MATH_OLMO_NAME: 2048 # Olmo-1B has 2048 hidden dim
        }
        
        for name in expert_names:
            expert_dim = expert_dims.get(name, max_hidden_dim) # Default to max_hidden_dim
            
            # Conceptual Expert Decoder (e.g., a simplified transformer block)
            # In reality, this would be the loaded AutoModelForCausalLM without its embedding/head
            self.experts[name] = torch.nn.Linear(expert_dim, expert_dim) # Dummy layer
            
            # Project-In layer: max_hidden_dim -> expert_dim
            self.proj_ins[name] = torch.nn.Linear(max_hidden_dim, expert_dim)
            
            # Project-Out layer: expert_dim -> max_hidden_dim
            self.proj_outs[name] = torch.nn.Linear(expert_dim, max_hidden_dim)
            
        # 3. Sequence-Level Router (conceptual)
        # The paper suggests a router network (MLP) for sequence-level routing.
        # For PPL-based routing, this would be external logic.
        # For a trainable router, it would be an MLP.
        self.router = torch.nn.Linear(max_hidden_dim, len(expert_names)) # Maps avg embedding to expert scores
        
        # 4. Shared Head Layer (conceptual)
        # Maps combined expert output back to vocabulary probabilities.
        self.shared_head = torch.nn.Linear(max_hidden_dim, vocab_size)
        
    def forward(self, input_ids):
        # 1. Get token embeddings from shared embedding layer
        embeddings = self.shared_embedding(input_ids)
        
        # 2. Average token embeddings for sequence-level routing
        avg_embedding = embeddings.mean(dim=1) # [batch_size, max_hidden_dim]
        
        # 3. Router computes weights (conceptual: using trainable router)
        router_logits = self.router(avg_embedding)
        expert_weights = torch.softmax(router_logits, dim=-1) # [batch_size, num_experts]
        
        # Select top-K experts (conceptual)
        # For simplicity, we'll use all experts with their weights here.
        # In a real MoE, only top-K would be selected and processed.
        
        combined_expert_output = torch.zeros_like(embeddings) # [batch_size, seq_len, max_hidden_dim]
        
        for i, name in enumerate(self.expert_names):
            # Apply Project-In
            proj_in_output = self.proj_ins[name](embeddings)
            
            # Process with expert decoder
            expert_output = self.experts[name](proj_in_output)
            
            # Apply Project-Out
            proj_out_output = self.proj_outs[name](expert_output)
            
            # Weight and combine outputs
            # Expand expert_weights to match sequence length for element-wise multiplication
            weighted_output = proj_out_output * expert_weights[:, i].unsqueeze(1).unsqueeze(2)
            combined_expert_output += weighted_output
            
        # 4. Feed combined representation into shared head layer
        logits = self.shared_head(combined_expert_output)
        
        return logits

# --- Conceptual Usage ---
heterogeneous_expert_names = [
    CODE_EXPERT_NAME,
    KNOWLEDGE_EXPERT_NAME,
    BASE_MODEL_NAME, # Base model also acts as an expert
    MATH_TINYLAMA_NAME # Heterogeneous expert
]

moe_hetero_model = HeterogeneousMoE(BASE_MODEL_NAME, heterogeneous_expert_names)
print("\nConceptual Heterogeneous MoE Model Structure:")
print(moe_hetero_model)

# Simulate an input
dummy_input_ids = torch.randint(0, 50257, (1, 10)) # Batch size 1, sequence length 10
conceptual_output_logits = moe_hetero_model(dummy_input_ids)
print(f"\nConceptual output logits shape: {conceptual_output_logits.shape}")
print("Conceptual Heterogeneous Model Merging setup complete.")


## 5. Pushing Merged Models to Hugging Face Hub (Conceptual)

After successfully merging models, you might want to push your new Mixture-of-Experts model to the Hugging Face Hub for sharing, versioning, or further deployment. This section conceptually outlines how you would do that.


In [ ]:
```python
# --- Conceptual Model Pushing ---
# In a real scenario, after `merge_models` successfully returns a merged model,
# you would save it and then push it to the Hugging Face Hub.
# Example (conceptual):
# merged_model.save_pretrained("my-merged-moe-model")
# tokenizer.save_pretrained("my-merged-moe-model")
# 
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="my-merged-moe-model",
#     repo_id="your-username/my-merged-moe-model", # Replace with your Hugging Face username and desired repo name
#     repo_type="model",
# )
# print("Conceptual: Merged model pushed to Hugging Face Hub.")
```


## 6. Conclusion

This notebook has provided a conceptual implementation of the model merging techniques described in the "MergeME" paper. It covers homogeneous merging using Dare/Ties, perplexity-based routing heuristics, architectural considerations for heterogeneous model merging, and conceptual Hugging Face integration.

To run this notebook with actual models, you would need to:
1.  Install `torch`, `transformers`, and `mergekit` in a compatible Python environment.
2.  Replace placeholder model names with actual Hugging Face model paths.
3.  Uncomment and adapt the model loading and merging sections to your specific setup and available resources.
4.  Implement actual model loading for `calculate_perplexity` and `HeterogeneousMoE` to enable full functionality.